## Installation boto3 dans jupyter

In [1]:
# L'image jupyter/scipy-notebook ne contient pas boto3 par défaut.
# On l'installe dans le kernel courant.
%pip install boto3==1.34.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 3.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 5.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.0/82.0 kB 5.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## Connexion a MinIO

In [5]:
import boto3
# Note importante : depuis le conteneur Jupyter, MinIO est accessible
# par son nom de service "minio", PAS par "localhost".
# C'est la résolution DNS automatique du réseau Compose.

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="anfa-app-key",
    aws_secret_access_key="anfa-app-secret-2026",
    region_name="us-east-1",
)

# Verifier la liste des buckets
s3.list_buckets()

{'ResponseMetadata': {'RequestId': '18BC27DE241B0F7A',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '366',
   'content-type': 'application/xml',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18BC27DE241B0F7A',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '2046',
   'x-ratelimit-remaining': '2046',
   'x-xss-protection': '1; mode=block',
   'date': 'Wed, 24 Jun 2026 23:23:07 GMT'},
  'RetryAttempts': 0},
 'Buckets': [{'Name': 'anfa-raw',
   'CreationDate': datetime.datetime(2026, 6, 23, 13, 55, 30, 399000, tzinfo=tzlocal())}],
 'Owner': {'DisplayName': 'minio',
  'ID': '02d6176db174dc93cb1b899f7c6078f08654445fe8cf1b6ce98d8855f66bdbf4'}}

## Lister les objets du bucket

In [6]:
reponse = s3.list_objects_v2(Bucket="anfa-raw", Prefix="referentiel/")
for obj in reponse.get("Contents", []):
    print(f"{obj['Key']} ({obj['Size']} octets)")

referentiel/arrets.csv (3728 octets)
referentiel/bus.csv (5812 octets)
referentiel/lignes.csv (933 octets)
referentiel/tarifs.csv (706 octets)


## Lire un CSV directement depuis MinIO avec pandas

In [7]:
import pandas as pd
from io import BytesIO

# Download le CSV des lignes en memoire
obj = s3.get_object(Bucket="anfa-raw", Key="referentiel/lignes.csv")
df_lignes = pd.read_csv(BytesIO(obj["Body"].read()))

df_lignes

,ligne_id,nom,terminus_depart,terminus_arrivee,nb_arrets,distance_km
0,L01,Adidogomé - Adawlato,Adidogomé Assiyéyé,Adawlato Marché,9,9.89
1,L02,Agoè Assiyéyé - Grand Marché,Agoè Assiyéyé Terminus,Adawlato Grand Marché,9,16.28
2,L03,Avédji - Adawlato,Avédji Limousine,Adawlato Marché,9,11.67
3,L04,Université de Lomé - Adawlato,Campus Universitaire,Adawlato Marché,8,5.65
4,L05,Hédzranawoé - Adawlato,Hédzranawoé Terminus,Adawlato Marché,8,7.37
5,L06,Bè - Tokoin Casablanca,Bè Kpota,Tokoin Casablanca,7,5.49
6,L07,Cacavéli - BIA Centre,Cacavéli Marché,BIA Centre,7,10.03
7,L08,Aéroport GTA - Adawlato,Aéroport GTA,Adawlato Marché,7,8.73
8,L09,Baguida - Adawlato,Baguida Terminus,Adawlato Marché,8,13.13
9,L10,Anfamé - Université de Lomé,Anfamé Terminus,Campus Universitaire,8,5.89


## Une analyse exploratoire

In [9]:
# Top 3 lignes les plus longues
df_lignes.nlargest(3, "distance_km")[["nom","nb_arrets","distance_km"]]

,nom,nb_arrets,distance_km
1,Agoè Assiyéyé - Grand Marché,9,16.28
8,Baguida - Adawlato,8,13.13
2,Avédji - Adawlato,9,11.67
